<a href="https://colab.research.google.com/github/pizzeman/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-23%20%E2%80%94%20Cleaning%20Clinic%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [1]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [2]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [3]:
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df.duplicated().sum())

(8, 6)
order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64
1


**What is wrong with this data?** List at least five specific problems:

1. There is duplicated data in the table
2. There is null data
3. Dollar sign in stored in the dataset
4. Font size is not consistent
5. qty is a signed float, which means it can accept negative values, which doesn't make much sense

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [4]:
clean = df.copy()

clean.drop_duplicates(inplace=True)
print(clean.duplicated().sum())
removed = 1   # TODO: how many duplicates were there?

log('duplicates', 'dropped exact duplicate rows', removed)

0
[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [5]:
# TODO: clean['price'] = ...
clean['price'] = clean['price'].astype(str)
clean['price'] = clean['price'].str.replace('$', '').str.replace(',','')
clean['price'] = clean['price'].str.strip()
clean['price'] = clean['price'].astype(float)

assert clean['price'].dtype == float
# TODO: log(...) -- note that price arrived as text
log('price', 'price arrived as text. stripped price', clean['price'])

[price] price arrived as text. stripped price (0     7.5
2     7.5
3    12.0
4    24.0
5     6.0
6     6.0
7    12.0
Name: price, dtype: float64 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [6]:
# TODO: clean['qty'] = pd.to_numeric(...)
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = df['qty'].isnull().sum()    # TODO: count of NaN quantities
negative = (df['qty'] < 0).sum()  # TODO: count of negative quantities

clean = clean[clean['qty'].notna() & (clean['qty'] > 0)].copy()
clean['qty'] = clean['qty'].astype(int)

# TODO: apply your decision, then log both separately
log('qty', 'removed NaN quantities', missing)
log('qty', 'removed negative quantities', negative)

[qty] removed NaN quantities (1 row(s))
[qty] removed negative quantities (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [7]:
print('before:', sorted(clean['category'].unique()))

# TODO: lowercase, strip, remove punctuation
clean['category'] = clean['category'].str.lower()
clean['category'] = clean['category'].str.strip()
clean['category'] = clean['category'].str.replace('-', '')

# TODO: CATEGORY_MAP = {...} for the judgment calls
CATEGORY_MAP = {
    'apparel': 'merch',
    'food': 'food',
    'merch': 'merch',
    'raingear': 'rain gear'
}
clean['category'] = clean['category'].replace(CATEGORY_MAP)

print('after: ', sorted(clean['category'].unique()))

before: ['Apparel', 'Food', 'Merch', 'food', 'rain-gear']
after:  ['food', 'merch', 'rain gear']


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [8]:
print('before:', sorted(clean['item'].dropna().unique()))

# TODO: lowercase, strip, remove punctuation
clean['item'] = clean['item'].str.lower().str.strip().str.replace('-', '').str.title()


# TODO: CATEGORY_MAP = {...} for the judgment calls
CATEGORY_MAP = {
  'Cheese Burger': 'Cheeseburger',
  'Foam Finger': 'Foam Finger',
  'Rain Poncho': 'Rain Poncho',
  'Uva Tshirt': 'UVA T-Shirt'
}
clean['item'] = clean['item'].replace(CATEGORY_MAP)

print('after: ', sorted(clean['item'].dropna().unique()))
# Keep the nan row

before: ['Cheeseburger', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
after:  ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt']


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [9]:
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce', format='mixed')
print(clean['ts'].dtype)
print('unparseable timestamps (NaT):', clean['ts'].isna().sum())
clean[['order_id', 'item', 'ts']]

clean['hour'] = clean['ts'].dt.hour
clean[['order_id', 'item', 'ts', 'hour']]

datetime64[ns]
unparseable timestamps (NaT): 1


,order_id,item,ts,hour
0,1,Cheeseburger,2026-09-05 12:03:00,12.0
2,2,Cheeseburger,2026-09-05 12:40:00,12.0
4,4,UVA T-Shirt,2026-09-05 13:05:00,13.0
6,6,Rain Poncho,NaT,NaN
7,7,NaN,2026-09-05 14:00:00,14.0


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [10]:
# TODO: assertions
assert clean.duplicated().sum() == 0, 'duplicates remain'
assert clean['price'].dtype == float, 'price is not numeric'
assert clean['qty'].min() >= 1, 'non-positive quantities remain'
assert pd.api.types.is_datetime64_any_dtype(clean['ts']), 'ts is not a datetime'
assert clean['category'].str.islower().all(), 'inconsistent category text'
print('clean:', clean.shape)

# TODO: clean['revenue'] = ...
clean['revenue'] = clean['price'] * clean['qty']
# TODO: print rows, units, revenue, distinct categories
print('revenue:', clean['revenue'].sum())
print('rows:', clean.shape[0])
print('units:', clean['qty'].sum())
print('distinct categories:', clean['category'].nunique())


clean: (5, 7)
revenue: 106.5
rows: 5
units: 10
distinct categories: 3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [11]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price,price arrived as text. stripped price,0 7.5 2 7.5 3 12.0 4 24.0 5 ...
2,qty,removed NaN quantities,1
3,qty,removed negative quantities,1


**The decision that mattered most:** removing duplicate rows affected the most

**Revenue with it:** \$114.00  **Revenue without it:** \$106.50

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [12]:
# Checkpoint
rows_after = 5            # TODO
revenue_after = 106.5         # TODO
biggest_decision = 'duplicate rows'    # TODO: which choice moved the number most
revenue_other_way = 114.00     # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 5
revenue: 106.5
decision that mattered: duplicate rows
revenue the other way: 114.0
